# Reaktive Maschinenagenten mit Mesa (v3.x)

In diesem Notebook wirst du:
- **Mesa 3.x** mit Visualisierungsunterstützung installieren,
- einen **reaktiven Agenten** implementieren, der die Temperatur einer Maschine überwacht,
- ein einfaches **Fabrikmodell** mit mehreren Maschinenagenten aufbauen,
- eine **interaktive Visualisierung** direkt im Notebook starten und verschiedene Parameter erkunden,
- das Modell um **Wartungs-** und **Auftragsagenten** erweitern.


## 1. Installation
Zunächst installieren wir Mesa **3.0.3** inklusive Visualisierungsunterstützung.


In [ ]:
#!pip install -U mesa[viz]==3.0.3 altair==5.2.0 networkx solara

## 2. Mesa-Grundlagen
Mesa-Modelle bestehen aus drei Hauptteilen:
- einer **Model**-Klasse, die den globalen Zustand verwaltet,
- einer oder mehreren **Agent**-Klassen, die das Verhalten einzelner Agenten definieren,
- einer optionalen **Visualisierung**, um das Modell während der Ausführung zu beobachten.

**Wichtige Änderungen in Mesa 3.x gegenüber v2.x:**
- `Agent.__init__` braucht keine `unique_id` mehr – sie wird automatisch vergeben.
- Den Scheduler (`RandomActivation`) gibt es nicht mehr → stattdessen `self.agents.shuffle_do("step")`.
- `model.schedule.agents` → `model.agents`.
- Agenten entfernen: `self.remove()` statt `self.model.schedule.remove(self)`.


## 3. Implementierung des `MachineAgent`
Der `MachineAgent` repräsentiert eine einzelne Maschine in einer Fabrik.
Er besitzt eine Temperatur und eine einfache reaktive Regel.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reaktiver Agent, der eine Maschine repräsentiert und deren Temperatur überwacht.

    Regel:
        temperature > threshold            -> Zustand = 'HOT'
        sonst                              -> Zustand = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)          # Mesa 3.x: kein unique_id-Argument
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False                # wird vom OrderAgent genutzt

    def sense_temperature(self):
        """Einfaches Sensormodell: Temperatur + zufälliges Rauschen."""
        if self.state != "HOT":
            noise = random.uniform(-1, 2)
            self.temperature += noise

    def decide(self):
        """Reaktive Entscheidungsregel, die nur auf der aktuellen Temperatur basiert."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        else:
            self.state = "OK"

    def act(self):
        """Maschine abkühlen, wenn sie überhitzt ist."""
        if self.state == "HOT":
            self.temperature -= 10

    def step(self):
        """Agentenschritt = wahrnehmen → entscheiden → handeln."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementierung des `FactoryModel`
Das Modell platziert mehrere Maschinen auf einem Gitter und aktiviert sie in
zufälliger Reihenfolge. Ein **DataCollector** verfolgt, wie viele Maschinen sich
in den Zuständen `"HOT"` befinden.


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

class FactoryModel(Model):
    """Einfaches Fabrikmodell mit einem Gitter aus Maschinenagenten."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisierung mit Mesa 3.x (SolaraViz)
Mesa 3.x verwendet **SolaraViz** als Visualisierungsframework.

Der Schlüssel ist die `post_process`-Funktion, die nach dem Zeichnen der Agenten
aufgerufen wird und beliebige Matplotlib-Anpassungen erlaubt – hier nutzen wir
sie, um die Temperatur als Text in jedes Kästchen zu schreiben.

⚠️ **Hinweis:** Falls die Anzeige im Notebook nicht direkt funktioniert,
speichere den Code in eine `.py`-Datei und starte sie mit `solara run datei.py`.


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component


# ── Agentendarstellung ─────────────────────────────────────────────────────────
def agent_portrayal(agent):
    color = {
        "HOT":  "red",
        "OK":   "green",
    }.get(agent.state, "gray")

    return {
        "color":  color,
        "size":   600,
        "marker": "s",
        "alpha":  0.9,
    }


# ── Nachbearbeitung: Temperaturtext und Grid ───────────────────────────────────
def post_process(ax):
    ax.figure.set_size_inches(6, 6)

    legend = ax.get_legend()
    if legend:
        legend.remove()

    for agent in model_instance.agents:
        if not isinstance(agent, MachineAgent):
            continue
        x, y = agent.pos
        ax.text(
            x, y,
            f"{agent.temperature:.1f}°C",
            ha="center", va="center",
            fontsize=7, color="white", fontweight="bold",
        )

    ax.grid(True, alpha=0.3)


# ── Komponenten ────────────────────────────────────────────────────────────────
space_component = make_space_component(
    agent_portrayal,
    post_process=post_process,
)

plot_component = make_plot_component(
    {"HotMachines": "red"}
)

# ── Modellparameter mit Schiebereglern ─────────────────────────────────────────
model_params = {
    "width":  10,
    "height": 10,
    "density": {
        "type":  "SliderFloat",
        "value": 0.3,
        "label": "Dichte",
        "min":   0.1,
        "max":   1.0,
        "step":  0.1,
    },
    "threshold": {
        "type":  "SliderInt",
        "value": 70,
        "label": "Temperatur-Schwelle",
        "min":   30,
        "max":   100,
        "step":  5,
    },
}

# ── Modellinstanz und Visualisierung ───────────────────────────────────────────
model_instance = FactoryModel(width=10, height=10, density=0.3, threshold=70)

page = SolaraViz(
    model_instance,
    components=[space_component, plot_component],
    model_params=model_params,
    name="Reaktive Maschinenagenten",
)
page


## 6. Simulation ohne Visualisierung testen
Falls du nur die Modelllogik überprüfen möchtest, kannst du das Modell auch direkt einige Schritte laufen lassen.


In [ ]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()


## 7. Erkundungsaufgaben (für das Praktikum)
Nutze das laufende Modell, um das Verhalten der reaktiven Agenten zu erkunden:

1. **Schwellenwert anpassen**
   - Starte mit `threshold=70` und probiere dann 60 oder 80.
   - Was passiert mit der Anzahl der `"HOT"`-Maschinen im Zeitverlauf?

2. **Dichte verändern**
   - Erhöhe oder verringere den `density`-Parameter.
   - Wie beeinflusst das den Gesamtzustand des Systems?

3. **Zustandslogik erweitern (optional)**
   - Füge weitere Zustände hinzu, z. B. `"KRITISCH"`, und mappe sie auf neue Farben in `agent_portrayal`.
   - Definiere eigene Temperaturbereiche für jeden Zustand.

4. **(Fortgeschritten) Wartungsagenten hinzufügen**
   - Erstelle einen zweiten Agententyp, der durch das Gitter wandert und überhitzte
     Maschinen „repariert", indem er deren Temperatur und Zustand zurücksetzt.

5. **(Fortgeschritten) Auftragsagenten hinzufügen**
   - Erstelle einen dritten Agententyp, der Aufträge durch das Gitter bewegt und
     selbstständig freie Maschinen sucht.
